# Gugak Stem Separation — EDA (Phase 2)

Audio-content EDA. **Source of truth = the manifest** (`manifests/parquet/*.parquet`); we never walk directories.

Phase 2 questions:
1. **Stem naming / strip rule** — validate `instrument` → `instrument_base` (multi-instrument stems).
2. **Sample rate** — confirm the 96 kHz (창작국악) set to resample.
3. **Listen** — do multi-instrument stems (피리 / 피리2 / 피리3) share a timbre? (manual check)
4. **Does summing stems give the master?** — residual test on 3 exemplar songs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
import IPython.display as ipd
import matplotlib.pyplot as plt

# Find repo root by walking up until we see the manifest (notebook cwd-agnostic).
def find_root(start: Path | None = None) -> Path:
    p = Path.cwd() if start is None else start
    for cand in [p, *p.parents]:
        if (cand / 'manifests' / 'songs.parquet').exists():
            return cand
    raise FileNotFoundError('repo root (with manifests/) not found above cwd')

ROOT = find_root()
songs = pd.read_parquet(ROOT / 'manifests' / 'songs.parquet')
stems = pd.read_parquet(ROOT / 'manifests' / 'stems.parquet')
print('root :', ROOT)
print('songs:', songs.shape, '| stems:', stems.shape)

## 1. Validate the stem-naming / strip rule

The manifest already carries `instrument` (raw, e.g. `피리2`) and `instrument_base` (stripped, `피리`).
**EDA principle: validate a parsing heuristic against the real values before building on it.**
Gugak trap to rule out: numbers that are part of an instrument's *identity* (e.g. 12현 vs 25현 가야금),
not a player index. Below: every raw name with a digit, and the raw→base mapping it produced.

In [ ]:
digit_names = sorted(stems.loc[stems['instrument'].str.contains(r'[0-9]'), 'instrument'].unique())
print('raw instrument names containing a digit:')
print(' ', digit_names)

print('\nraw -> base (only where they differ), with counts:')
diff = stems[stems['instrument'] != stems['instrument_base']]
print(diff.groupby(['instrument', 'instrument_base']).size()
          .sort_values(ascending=False).to_string())

print('\nsongs with >1 stem sharing an instrument_base (true multi-instrument):')
dup = stems.groupby(['song_id', 'instrument_base']).size()
dup = dup[dup > 1].sort_values(ascending=False)
print(f'  {len(dup)} (song, base) groups; genres involved:',
      sorted(stems[stems.song_id.isin(dup.index.get_level_values(0))].genre_sub.unique()))
print(dup.head(10).to_string())

**Result:** every digit is a trailing single digit on a legitimate base instrument (아쟁2, 피리2, …) —
no identity-number trap (no `가야금25`). Multi-instrument songs are **100% 창작국악**. Strip rule is safe.

## 2. Sample rate — confirm the 96 kHz resample set

Everything is 48 kHz except a 창작국악 subset at 96 kHz. Confirm the exact count and that it's
genre-localized, from the manifest headers. (CLAUDE.md: ~130 files → resample to 48 kHz.)

In [ ]:
print('stem sample-rate counts:');   print(stems['sr'].value_counts().to_string())
print('\nmaster sample-rate counts:'); print(songs['master_sr'].value_counts().to_string())

hi = stems[stems['sr'] != 48000]
hi_master = songs[songs['master_sr'] != 48000]
print(f'\n96 kHz: {len(hi)} stems + {len(hi_master)} masters = {len(hi) + len(hi_master)} files'
      f'  across {hi.song_id.nunique()} songs')
print('genres of 96 kHz stems:', sorted(hi.genre_sub.unique()))

## 3. Listen: do 피리 / 피리2 / 피리3 share a timbre?

If they're the same source class (same instrument, different players), we **sum** them into one
target signal — the model can't (and shouldn't) split same-timbre players apart.

**Gotcha we hit and fixed:** gugak stems are *sparse* — an instrument may play only a small fraction
of the song (some here are active <10% of the time). A fixed time window often lands on silence, and
`ipd.Audio` divides by zero when normalizing an all-silent clip. So we **find each stem's loudest
window** and audition there. Note: `ipd.Audio` normalizes each clip, so all players sound equally
loud — compare true levels with the printed `peak`/`rms`, not by ear (levels span ~30 dB across stems).

In [ ]:
def load_active_excerpt(rel_path: str, seconds: float = 15.0, hop_s: float = 0.5):
    """Return the loudest `seconds`-long window of a WAV (float32, stereo), via a sliding-RMS scan.

    One full decode + an O(1)-per-window energy scan (cumulative sum). Fine for a handful of stems.
    """
    audio, sr = sf.read(ROOT / rel_path, dtype='float32', always_2d=True)  # (frames, ch)
    win = min(int(seconds * sr), len(audio))
    mono = audio.mean(axis=1).astype(np.float64)
    csum = np.concatenate([[0.0], np.cumsum(mono ** 2)])         # prefix energy
    hop = max(1, int(hop_s * sr))
    starts = range(0, len(audio) - win + 1, hop)
    energies = np.array([csum[s + win] - csum[s] for s in starts]) if len(audio) > win else np.array([0.0])
    best = list(starts)[int(energies.argmax())] if len(audio) > win else 0
    excerpt = audio[best:best + win]
    win_rms = float(np.sqrt(energies.max() / win)) if len(audio) > win else float(np.sqrt((mono ** 2).mean()))
    return excerpt, sr, best / sr, win_rms

def play(rel_path: str, label: str, seconds: float = 15.0):
    exc, sr, t0, win_rms = load_active_excerpt(rel_path, seconds)
    peak = float(np.abs(exc).max())
    if peak < 1e-4:
        print(f'{label:8}  SILENT everywhere (peak≈0) — no player'); return
    print(f'{label:8}  loudest @ {t0:6.1f}s  win_rms={win_rms:.4f}  peak={peak:.3f}')
    ipd.display(ipd.Audio(exc.T, rate=sr))   # .T -> (ch, frames); normalized per-clip

SONG = '0979_창작국악_창작국악'
piri = stems[(stems.song_id == SONG) & (stems.instrument_base == '피리')].sort_values('instrument')
for _, r in piri.iterrows():
    play(r.stem_path, r.instrument)

For contrast, listen across *different* instruments in the 관악 group (피리 vs 대금 vs 단소)
— distinct timbres the model **should** separate, unlike the 피리 copies above.

In [ ]:
gwan = (stems[(stems.song_id == SONG) & (stems.instrument_base.isin(['피리', '대금', '단소']))]
        .drop_duplicates('instrument_base').sort_values('instrument_base'))
for _, r in gwan.iterrows():
    play(r.stem_path, r.instrument_base)

## 4. Does summing the stems give you the master?  (3 exemplars)

We build training mixtures by **summing stems** (mixture := Σ stems, targets := stems —
self-consistent by construction). Separate question: is the *real master* that same signal? Here we
compare each master directly against its **naive equal-weight stem sum**.

**How to read the panels:**
- **Left** — waveform overlay, ~40 ms of a loud passage. Do master and Σstems trace the same shape?
- **Right** — every audio sample as a dot: **x = Σstems at that instant, y = master at that instant.**
  If master = sum, every dot lands on the diagonal `y = x`. A **tilted straight line** = linear but
  wrong level (a fader/volume fix); a **fuzzy cloud** = the master carries content the sum doesn't
  (reverb/FX). Annotation gives the residual for the naive sum vs the best per-stem linear fit.

Tiers (from the stratified sample — `scripts/residual_test.py`):
- **Tier 1 `0718_민속악_민요`** — master **is** the naive sum (−33 dB).
- **Tier 2 `0151_정악_궁중음악`** — naive sum off (−2.8 dB), but a per-stem **fader** fit recovers it
  (−32 dB) → *linear, wrong levels.*
- **Tier 3 `0885_창작국악`** — even the best linear fit stalls at −15 dB → *nonlinear / added FX.*

Stems are heterophonic → collinear, so the fitted fader weights can be non-physical — trust the
**residual dB**, not the weights.

**Takeaway:** summing stems is safe for *training* (it defines our mixture); the real master is a
**secondary, domain-shifted eval**, not the primary metric — and we don't try to invert master→stems.

In [ ]:
import sys
from matplotlib.colors import LogNorm
sys.path.insert(0, str(ROOT))
from src.data.audio import (read_audio, to_mono, estimate_lag, fit_linear_mix,
                            prefix_energy, loudest_window, rms, to_db)

sp = stems.groupby('song_id')['stem_path'].apply(list).to_dict()
songs_i = songs.set_index('song_id')
EXEMPLARS = [
    ('0718_민속악_민요', 'Tier 1 · master IS the naive sum'),
    ('0151_정악_궁중음악', 'Tier 2 · linear, but needs per-stem faders'),
    ('0885_창작국악_창작국악', 'Tier 3 · master \u2260 any linear stem mix'),
]
C_MASTER, C_SUM = '#0072B2', '#D55E00'   # Okabe-Ito blue / vermillion (CVD-safe)
SR = 48000

fig, axes = plt.subplots(3, 2, figsize=(11, 12), gridspec_kw={'width_ratios': [1.15, 1]})
for row, (sid, title) in enumerate(EXEMPLARS):
    S = [read_audio(ROOT / p)[0].astype('float64') for p in sp[sid]]
    L = min(len(s) for s in S); S = [s[:L] for s in S]
    naive = np.sum(S, axis=0)
    master = read_audio(ROOT / songs_i.loc[sid, 'master_path'])[0].astype('float64')
    d = estimate_lag(master, naive, SR)
    if d >= 0:
        master = master[d:]
    else:
        S = [s[-d:] for s in S]; naive = np.sum(S, axis=0)
    Lc = min(len(master), len(naive)); master = master[:Lc]; naive = naive[:Lc]; S = [s[:Lc] for s in S]

    resid_naive = to_db(rms(master - naive) / rms(master))
    _, recon = fit_linear_mix(S, master)
    resid_ps = to_db(rms(master - recon) / rms(master))
    mm, ms = to_mono(master), to_mono(naive)

    # left: waveform overlay, ~40 ms of a loud passage (short window so shapes are visible)
    win = int(0.040 * SR)
    start = loudest_window(prefix_energy(mm), win, hop=int(0.005 * SR))
    t = np.arange(win) / SR * 1000
    ax = axes[row, 0]
    ax.plot(t, ms[start:start + win], color=C_SUM, lw=1.3, alpha=0.85, label='Σ stems (naive sum)')
    ax.plot(t, mm[start:start + win], color=C_MASTER, lw=1.3, alpha=0.85, label='master')
    ax.set_title(title, fontsize=11, loc='left', weight='bold')
    ax.set_xlabel('ms'); ax.set_ylabel('amplitude')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(True, lw=0.4, alpha=0.3)
    if row == 0:
        ax.legend(frameon=False, fontsize=9, loc='upper right')

    # right: master vs naive sum, sample-by-sample density
    step = max(1, len(mm) // 200_000)
    x, y = ms[::step], mm[::step]
    lim = float(np.percentile(np.abs(np.concatenate([x, y])), 99.9))
    ax = axes[row, 1]
    ax.hexbin(x, y, gridsize=70, cmap='Blues', norm=LogNorm(), extent=(-lim, lim, -lim, lim))
    ax.plot([-lim, lim], [-lim, lim], color='#444', lw=1.0, ls='--', label='y = x  (master = sum)')
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal')
    ax.set_xlabel('Σ stems sample'); ax.set_ylabel('master sample')
    ax.spines[['top', 'right']].set_visible(False)
    ax.text(0.04, 0.96, f'naive sum: {resid_naive:+.1f} dB\nbest linear fit: {resid_ps:+.1f} dB',
            transform=ax.transAxes, va='top', ha='left', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='#ccc', alpha=0.9))
    if row == 0:
        ax.legend(frameon=False, fontsize=8, loc='lower right')

fig.suptitle('Does summing the stems give you the master?  (residual rel. master; lower dB = closer)',
             fontsize=12.5, weight='bold', y=0.997)
fig.tight_layout()
fig.savefig(ROOT / 'notebooks' / 'fig_master_vs_sum.png', dpi=110, bbox_inches='tight')
plt.show()

## 5. Dataset overview (deliverable)

Manifest-only stats (no audio decode) for the instructor briefing: 4-stem class balance,
genre composition, instrument prevalence, and ensemble size. 4-stem groups are color-coded
consistently (관악/찰현/발현/타악). Saves `notebooks/fig_dataset_overview.png`.

Headlines: **4-stem balance is healthy** (관악 31% / 찰현 25% / 발현 25% / 타악 20% by hours);
판소리 (269 songs) and 창작국악 (187) dominate; the "big 6" 대금·피리·해금·가야금·거문고·아쟁 appear in
~600–700 songs each; ensemble size is bimodal (2 = 산조 solo+장구, ~7 = typical, up to 18 = 창작국악).

In [ ]:
import koreanize_matplotlib  # noqa: F401  (registers NanumGothic so Korean labels render)
from matplotlib.patches import Patch

H = lambda s: s.sum() / 3600
GROUP_ORDER = ['관악', '찰현', '발현', '타악']
GC = {'관악': '#0072B2', '찰현': '#D55E00', '발현': '#009E73', '타악': '#E69F00'}  # CVD-safe
ACCENT = '#4C72B0'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# A: 4-stem class balance (hours)
ax = axes[0, 0]
sg = stems.groupby('stem_group_4').dur_s.apply(H).reindex(GROUP_ORDER); tot = sg.sum()
bars = ax.bar(GROUP_ORDER, sg.values, color=[GC[g] for g in GROUP_ORDER], width=0.7)
for b, v in zip(bars, sg.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.0f} h\n{v / tot * 100:.0f}%',
            ha='center', va='bottom', fontsize=10)
ax.set_title('4-stem class balance (stem-hours)', fontsize=12, weight='bold', loc='left')
ax.set_ylabel('hours'); ax.set_ylim(0, sg.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', lw=0.4, alpha=0.3)

# B: genre composition (songs, hours annotated)
ax = axes[0, 1]
gsong = songs.genre_sub.value_counts(); ghours = stems.groupby('genre_sub').dur_s.apply(H)
order = gsong.index[::-1]
ax.barh(order, gsong[order].values, color=ACCENT, height=0.7)
for i, g in enumerate(order):
    ax.text(gsong[g] + 3, i, f'{gsong[g]}  ·  {ghours[g]:.0f} h', va='center', fontsize=9)
ax.set_title('Songs per genre (·  stem-hours)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('songs'); ax.set_xlim(0, gsong.max() * 1.28)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', lw=0.4, alpha=0.3)

# C: instrument prevalence (top 12), colored by 4-stem group
ax = axes[1, 0]
ib = (stems.groupby('instrument_base')
            .agg(songs=('song_id', 'nunique'), group=('stem_group_4', 'first'))
            .sort_values('songs', ascending=False).head(12).iloc[::-1])
ax.barh(ib.index, ib.songs.values, color=[GC[g] for g in ib.group], height=0.72)
for i, v in enumerate(ib.songs.values):
    ax.text(v + 6, i, str(v), va='center', fontsize=9)
ax.set_title('Instruments by prevalence (top 12, # songs present)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('songs present'); ax.set_xlim(0, ib.songs.max() * 1.15)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='x', lw=0.4, alpha=0.3)
ax.legend(handles=[Patch(color=GC[g], label=g) for g in GROUP_ORDER],
          frameon=False, fontsize=9, ncol=4, loc='lower right')

# D: ensemble size (stems per song)
ax = axes[1, 1]
ns = songs.n_stems
ax.hist(ns, bins=np.arange(0.5, ns.max() + 1.5, 1), color=ACCENT, alpha=0.9, rwidth=0.9)
med = ns.median(); ax.axvline(med, color='#D55E00', lw=1.5, ls='--')
ax.text(med + 0.3, ax.get_ylim()[1] * 0.9, f'median {med:.0f}', color='#D55E00', fontsize=9)
ax.set_title('Ensemble size (stems per song)', fontsize=12, weight='bold', loc='left')
ax.set_xlabel('stems in song'); ax.set_ylabel('songs')
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', lw=0.4, alpha=0.3)

fig.suptitle(f'Gugak dataset overview — {len(songs)} songs · {len(stems)} stems · '
             f'{H(stems.dur_s):.0f} stem-hours · split 721/91/91',
             fontsize=13.5, weight='bold', y=0.998)
fig.tight_layout()
fig.savefig(ROOT / 'notebooks' / 'fig_dataset_overview.png', dpi=110, bbox_inches='tight')
plt.show()

## 6. Cross-dataset stem-group distribution (11-group working taxonomy)

Total **duration (hours) per stem group, stacked by dataset** — how much material each of the two sets
contributes to each class of the current scheme. Bars sorted **most → least total duration**; the caption
above each bar is the **source-file count from each dataset** (`ens:` = 71955 stems, `solo:` = 71470 clips).
Saves `figs/instrument_distribution_2.png` (gitignored — regenerable from this cell).

**Source of truth:** `manifests/parquet/source_manifest.parquet` ⋈ `configs/stem_taxonomy.yaml` (the manifest
already carries the resolved `stem_group`). No directory walking, no ad-hoc mapping — this supersedes the
earlier version of this section, which predated the taxonomy and folded 71470 vocals and obscure winds
into 기타 and its percussion into 타악기.

**Scope:** 903 masters excluded (mixes, not stems). The **222 held-aside clips** — `stem_group: null`,
i.e. 징 · 퉁소 and the 문묘제례악 ritual instruments, deliberately undecided pending expert consultation —
are excluded, and the subtitle says so. All splits pooled (this is a corpus-overview figure).
`pitched_percussion` and `voice` are displayed as 유율타악 / 성악 for a consistent Korean axis; the
canonical config keys are unchanged.

**Headline:** 71955 (369 h / 5,767 stems) carries essentially all the duration; 71470 (39 h / 9,723 clips)
adds **breadth, not bulk** — it roughly doubles-to-triples the independent source count in the melodic
classes (해금 +1,291, 가야금 +1,356, 피리 +916) for ~+10% audio, which is precisely why its value to the
mixer is *draw diversity* rather than extra training hours. Two structural facts are visible directly:
**성악 is solo-only** (`ens:0`) — 71955 has no vocal stem anywhere, including 판소리 — and the two smallest
classes, 양금 (6.6 h) and 유율타악 (2.9 h), are the ones flagged for the prof.

In [ ]:
# --- Stem-group duration by dataset (71955 ensemble stems + 71470 solo clips) ---
# Manifest-driven (repo rule: never walk directories). Self-contained — independent of the
# earlier cells, whose songs/stems parquets were removed in the 2026-07-24 restructure.
# Saves figs/instrument_distribution_2.png.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # noqa: F401  (registers NanumGothic so Korean labels render)


# Locate the repo root so the cell runs from any working directory
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upwards from `start` until the directory holding pyproject.toml is found.

    Args:
        start: directory to begin the search from; defaults to the current working directory.
    """
    search_start = Path.cwd() if start is None else start
    for candidate in [search_start, *search_start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('repo root (pyproject.toml) not found above cwd')


REPO_ROOT = find_repo_root()
DATASET_ENSEMBLE, DATASET_SOLO = '71955', '71470'

# Display-only labels for the two groups whose canonical keys are English. The keys in
# configs/stem_taxonomy.yaml stay `pitched_percussion` / `voice`; these are axis labels only
# (all-Korean ticks, and `pitched_percussion` is too wide to fit its slot).
DISPLAY_LABEL = {'pitched_percussion': '유율타악', 'voice': '성악'}

# Keep only real sources. Masters are mixes, not stems. The 222 clips with a null stem_group
# are HELD ASIDE (징·퉁소·문묘제례악 ritual instruments) — a pending taxonomy backlog rather than
# a stem class, so they are excluded here and the exclusion is declared in the subtitle.
source_manifest = pd.read_parquet(REPO_ROOT / 'manifests' / 'parquet' / 'source_manifest.parquet')
non_master = source_manifest[source_manifest['role'] != 'master']
sources = non_master[non_master['stem_group'].notna()]
held_aside_count = int(non_master['stem_group'].isna().sum())

# Aggregate duration and item count per (stem group x dataset)
grouped = (sources.groupby(['stem_group', 'dataset'])
                  .agg(seconds=('out_duration', 'sum'), items=('file_id', 'size'))
                  .unstack(fill_value=0))
group_order = (grouped[('seconds', DATASET_ENSEMBLE)] + grouped[('seconds', DATASET_SOLO)]) \
    .sort_values(ascending=False).index.tolist()
hours_ensemble = (grouped[('seconds', DATASET_ENSEMBLE)] / 3600).reindex(group_order).values
hours_solo = (grouped[('seconds', DATASET_SOLO)] / 3600).reindex(group_order).values
items_ensemble = grouped[('items', DATASET_ENSEMBLE)].reindex(group_order).values
items_solo = grouped[('items', DATASET_SOLO)].reindex(group_order).values

# Stacked bars, two CVD-safe Okabe-Ito hues (teal-green / reddish-purple)
COLOR_ENSEMBLE, COLOR_SOLO = '#009E73', '#CC79A7'
INK, MUTED = '#222222', '#666666'
bar_positions = np.arange(len(group_order))
bar_totals = hours_ensemble + hours_solo

fig, ax = plt.subplots(figsize=(12, 6.5))
ax.bar(bar_positions, hours_ensemble, 0.68, color=COLOR_ENSEMBLE, edgecolor='white', linewidth=0.8,
       label=f'71955 ensemble stems  ·  {hours_ensemble.sum():.0f} h / {items_ensemble.sum():,} stems')
ax.bar(bar_positions, hours_solo, 0.68, bottom=hours_ensemble, color=COLOR_SOLO,
       edgecolor='white', linewidth=0.8,
       label=f'71470 solo clips  ·  {hours_solo.sum():.0f} h / {items_solo.sum():,} clips')

# Per-bar caption: how many source files each dataset contributes to this stem group
for position, total, n_ensemble, n_solo in zip(bar_positions, bar_totals, items_ensemble, items_solo):
    ax.text(position, total + bar_totals.max() * 0.018,
            f'ens:{n_ensemble:,}\nsolo:{n_solo:,}',
            ha='center', va='bottom', fontsize=8.5, color=MUTED, linespacing=1.35)

# Axes, titles, cosmetics
ax.set_xticks(bar_positions)
ax.set_xticklabels([DISPLAY_LABEL.get(group, group) for group in group_order],
                   fontsize=11.5, color=INK)
ax.set_ylabel('total duration (hours)', fontsize=11, color=INK)
ax.set_ylim(0, bar_totals.max() * 1.20)
fig.suptitle('Stem-group duration by dataset — 11-group working taxonomy',
             x=0.012, ha='left', fontsize=13.5, weight='bold', color=INK, y=0.99)
ax.set_title(f'bars sorted by total hours  ·  caption = source-file count per dataset  ·  '
             f'{held_aside_count} held-aside clips excluded',
             loc='left', fontsize=9, color=MUTED, pad=10)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
ax.tick_params(colors=MUTED)
ax.grid(axis='y', lw=0.4, alpha=0.35)
ax.set_axisbelow(True)
fig.tight_layout()

(REPO_ROOT / 'figs').mkdir(exist_ok=True)
fig.savefig(REPO_ROOT / 'figs' / 'instrument_distribution_2.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'saved figs/instrument_distribution_2.png  ·  {len(group_order)} groups  ·  '
      f'71955 {hours_ensemble.sum():.1f} h / {items_ensemble.sum():,}  ·  '
      f'71470 {hours_solo.sum():.1f} h / {items_solo.sum():,}  ·  '
      f'{held_aside_count} held-aside excluded')

### 6b. …annotated with instrument membership

Same bars as above, plus a **membership table** naming the canonical instruments that actually feed each
stem group, with each instrument's source count as `71955 + 71470`. Saves
`figs/instrument_distribution_3.png` — a superset of `_2`, kept as a separate figure so both the clean
and the annotated version survive.

Membership is **observed from the manifest, not read off the taxonomy** — an instrument appears here only
if it has files. (The two differ: `configs/stem_taxonomy.yaml` declares 방울 under 타악기 on inferred
grounds, and the manifest confirms 7 real stems.)

**What the table is for:** six of the eleven groups are single-instrument, so the scheme's actual
compromises live in the other five. 타악기 is a 10-instrument catch-all dominated by 장구 (571+445) with
five ensemble-only members; 기타 is four pitched winds, not a miscellany; 성악 and 유율타악 are the two
additions to the publisher's scheme. The greyed final row lists the **222 held-aside clips** excluded from
the bars — 징 (127) and 퉁소 (53) are 81% of that backlog.

In [ ]:
# --- Same distribution, annotated with per-group instrument membership ---
# Self-contained variant of the cell above: identical bars, plus a membership table listing
# which canonical instruments actually feed each stem group (manifest-observed, not merely
# declared in the taxonomy). Saves figs/instrument_distribution_3.png.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib  # noqa: F401  (registers NanumGothic so Korean labels render)


# Locate the repo root so the cell runs from any working directory
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upwards from `start` until the directory holding pyproject.toml is found.

    Args:
        start: directory to begin the search from; defaults to the current working directory.
    """
    search_start = Path.cwd() if start is None else start
    for candidate in [search_start, *search_start.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('repo root (pyproject.toml) not found above cwd')


REPO_ROOT = find_repo_root()
DATASET_ENSEMBLE, DATASET_SOLO = '71955', '71470'
DISPLAY_LABEL = {'pitched_percussion': '유율타악', 'voice': '성악'}
COLOR_ENSEMBLE, COLOR_SOLO = '#009E73', '#CC79A7'   # Okabe-Ito, CVD-safe
INK, MUTED, FAINT = '#222222', '#666666', '#999999'

# Split the manifest into the charted sources and the held-aside backlog
source_manifest = pd.read_parquet(REPO_ROOT / 'manifests' / 'parquet' / 'source_manifest.parquet')
non_master = source_manifest[source_manifest['role'] != 'master']
sources = non_master[non_master['stem_group'].notna()]
held_aside = non_master[non_master['stem_group'].isna()]

# Aggregate duration and item count per (stem group x dataset)
grouped = (sources.groupby(['stem_group', 'dataset'])
                  .agg(seconds=('out_duration', 'sum'), items=('file_id', 'size'))
                  .unstack(fill_value=0))
group_order = (grouped[('seconds', DATASET_ENSEMBLE)] + grouped[('seconds', DATASET_SOLO)]) \
    .sort_values(ascending=False).index.tolist()
hours_ensemble = (grouped[('seconds', DATASET_ENSEMBLE)] / 3600).reindex(group_order).values
hours_solo = (grouped[('seconds', DATASET_SOLO)] / 3600).reindex(group_order).values
items_ensemble = grouped[('items', DATASET_ENSEMBLE)].reindex(group_order).values
items_solo = grouped[('items', DATASET_SOLO)].reindex(group_order).values

# Per-group instrument membership, ordered by how much each instrument contributes.
# Counts are rendered "ensemble+solo" so a reader can see both the makeup of a group and
# which side of the corpus each member comes from (e.g. 소리북 269+0 is ensemble-only).
members_by_group = (sources.groupby(['stem_group', 'instrument_canonical', 'dataset'])
                           .size().unstack(fill_value=0)
                           .reindex(columns=[DATASET_ENSEMBLE, DATASET_SOLO], fill_value=0))
members_by_group['total'] = members_by_group.sum(axis=1)


def format_members(group: str) -> str:
    """Render one group's instrument membership as a single ' · '-separated line.

    Args:
        group: stem_group key to describe.
    """
    rows = members_by_group.loc[group].sort_values('total', ascending=False)
    return '  ·  '.join(f'{instrument} {row[DATASET_ENSEMBLE]}+{row[DATASET_SOLO]}'
                        for instrument, row in rows.iterrows())


# Held-aside instruments are 71470-only, so a plain count reads better than "0+n"
held_aside_members = held_aside.groupby('instrument_canonical').size().sort_values(ascending=False)
held_aside_line = '  ·  '.join(f'{name} {count}' for name, count in held_aside_members.items())

# Layout: bars on top, membership table beneath, sharing the figure width
fig = plt.figure(figsize=(15, 9.5))
grid = fig.add_gridspec(2, 1, height_ratios=[2.5, 1.25], hspace=0.14, top=0.93, bottom=0.04)
ax = fig.add_subplot(grid[0])
ax_table = fig.add_subplot(grid[1])

bar_positions = np.arange(len(group_order))
bar_totals = hours_ensemble + hours_solo
ax.bar(bar_positions, hours_ensemble, 0.68, color=COLOR_ENSEMBLE, edgecolor='white', linewidth=0.8,
       label=f'71955 ensemble stems  ·  {hours_ensemble.sum():.0f} h / {items_ensemble.sum():,} stems')
ax.bar(bar_positions, hours_solo, 0.68, bottom=hours_ensemble, color=COLOR_SOLO,
       edgecolor='white', linewidth=0.8,
       label=f'71470 solo clips  ·  {hours_solo.sum():.0f} h / {items_solo.sum():,} clips')

# Per-bar caption: how many source files each dataset contributes to this stem group
for position, total, n_ensemble, n_solo in zip(bar_positions, bar_totals, items_ensemble, items_solo):
    ax.text(position, total + bar_totals.max() * 0.018,
            f'ens:{n_ensemble:,}\nsolo:{n_solo:,}',
            ha='center', va='bottom', fontsize=8.5, color=MUTED, linespacing=1.35)

ax.set_xticks(bar_positions)
ax.set_xticklabels([DISPLAY_LABEL.get(group, group) for group in group_order],
                   fontsize=11.5, color=INK)
ax.set_ylabel('total duration (hours)', fontsize=11, color=INK)
ax.set_ylim(0, bar_totals.max() * 1.20)
ax.set_title(f'bars sorted by total hours  ·  caption = source-file count per dataset  ·  '
             f'{len(held_aside)} held-aside clips excluded',
             loc='left', fontsize=9, color=MUTED, pad=10)
ax.legend(frameon=False, fontsize=10, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#cccccc')
ax.tick_params(colors=MUTED)
ax.grid(axis='y', lw=0.4, alpha=0.35)
ax.set_axisbelow(True)

# Membership table: one row per bar, in bar order, then the excluded backlog
ax_table.axis('off')
table_rows = [(DISPLAY_LABEL.get(group, group), format_members(group), INK, INK)
              for group in group_order]
table_rows.append((f'held aside ({len(held_aside)})', held_aside_line, FAINT, FAINT))

# Rows are laid out from the top: header at y=1, then one row every `row_height`, with the
# divider sitting midway between header and first row. row_height is chosen so the last row
# still clears the axes floor.
row_height = 1.0 / (len(table_rows) + 1.5)
ax_table.text(0.0, 1.0, 'stem group', fontsize=9, weight='bold', color=MUTED, va='center')
ax_table.text(0.135, 1.0, 'member instruments   (count in 71955 + count in 71470)',
              fontsize=9, weight='bold', color=MUTED, va='center')
ax_table.axhline(1.0 - row_height * 0.6, color='#bbbbbb', lw=0.9)
for row_index, (group_label, member_line, name_color, line_color) in enumerate(table_rows):
    row_y = 1.0 - row_height * (row_index + 1.5)
    ax_table.text(0.0, row_y, group_label, fontsize=9.5, weight='bold', color=name_color, va='center')
    ax_table.text(0.135, row_y, member_line, fontsize=8.5, color=line_color, va='center')
# Separate the excluded backlog from the eleven real classes
ax_table.axhline(1.0 - row_height * (len(table_rows) - 0.1), color='#e5e5e5', lw=0.8)
ax_table.set_xlim(0, 1)
ax_table.set_ylim(0, 1.06)

fig.suptitle('Stem-group duration by dataset, with instrument membership — 11-group working taxonomy',
             x=0.012, ha='left', fontsize=14, weight='bold', color=INK, y=0.975)

(REPO_ROOT / 'figs').mkdir(exist_ok=True)
fig.savefig(REPO_ROOT / 'figs' / 'instrument_distribution_3.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'saved figs/instrument_distribution_3.png  ·  {len(group_order)} groups  ·  '
      f'{len(members_by_group)} instruments charted  ·  '
      f'{len(held_aside_members)} held aside ({len(held_aside)} clips)')